# Persian AI-Text Detection — Colab Retraining Pipeline

The local machine this project runs on is CPU-only with almost no free disk, so the
first round of models (SVM, Logistic Regression, Naive Bayes, Random Forest,
Gradient Boosting, BiLSTM) were trained **without any hyperparameter tuning** and,
for the tree models, on a reduced feature set just to make them finish. That is the
main reason they make bad mistakes on real-world text: they were never actually
tuned, and the classic-ML feature space (word 3-grams only, 9.3M dimensions) is
prone to memorizing training n-grams instead of generalizing.

The GPT-2 model (`gpt2_transformer`) started out better because it builds on real
pretrained language understanding, but it was also fine-tuned under the same CPU
constraints — on a 6,000/1,500 subsample, only 64 tokens of context, and only the
last 4 of 12 transformer blocks unfrozen, just to make it finish in reasonable
time. This notebook fine-tunes it properly too.

This notebook retrains everything:
- a saner TF-IDF space (unigrams + bigrams + trigrams, capped vocabulary) for the
  classic ML models
- real hyperparameter search (`RandomizedSearchCV` / `GridSearchCV`) for every
  classic model, on Colab's compute instead of the local box's
- a fixed train / validation / test split — earlier the BiLSTM (and GPT-2) picked
  their "best" checkpoint by peeking at the test set, which quietly inflated their
  reported scores; here validation is a separate slice and the test set is
  touched exactly once, at the end, per model
- a bigger BiLSTM, trained on GPU, with early stopping and weight decay
- GPT-2 fine-tuned on the **full** dataset, all 12 transformer blocks unfrozen,
  128 tokens of context, GPU + mixed precision, plus a better tokenization
  approach (see the GPT-2 section for why)

**Runtime:** `Runtime > Change runtime type > GPU` before running — the BiLSTM and
GPT-2 sections both use it; everything else is CPU-bound regardless.

**What to bring back to the project:** at the end this notebook saves every
artifact plus a `models_colab_update.json` fragment and zips them for download.
Drop the files into the project root, merge the JSON fragment into `models.json`
(it reuses the same ids, so just replace those entries), and restart `server.py`.


## 1. Setup

In [ ]:
!pip install -q shekar "transformers>=4.40"

import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("No GPU — go to Runtime > Change runtime type > GPU before continuing.")


In [ ]:
import json
import math
import multiprocessing
import re
import time
from collections import Counter

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from joblib import Parallel, delayed
from scipy.sparse import csr_matrix, hstack
from scipy.stats import loguniform, randint
from sklearn.compose import ColumnTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


## 2. Load the dataset

Bring in **`processedDataset.csv`** from the project — it's the already
balanced/shuffled 21,092-row dataset (`text`, `aiGenerated`) the local models were
trained on, so results stay comparable. It's ~160MB: the most reliable way to get
it here is uploading it to Google Drive once and mounting Drive below (a raw
`files.upload()` for a file this size tends to time out in the browser).

If you'd rather rebuild it from scratch, upload `persian_ai_generated_final.csv`
and `blogs.csv` instead and use the fallback cell.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Edit this to wherever you uploaded processedDataset.csv in your Drive.
DATASET_PATH = "/content/drive/MyDrive/AITextDetection/processedDataset.csv"

df = pd.read_csv(DATASET_PATH)
print(df.shape)
print(df["aiGenerated"].value_counts())
df.head(3)


In [ ]:
# --- Fallback: rebuild processedDataset.csv from the two raw source files ---
# Only run this if you uploaded persian_ai_generated_final.csv + blogs.csv instead.
#
# ai_df = pd.read_csv("/content/drive/MyDrive/AITextDetection/persian_ai_generated_final.csv", encoding="utf-8-sig")
# human_df = pd.read_csv("/content/drive/MyDrive/AITextDetection/blogs.csv", encoding="utf-8-sig")
# human_df = human_df.sample(len(ai_df), random_state=RANDOM_SEED, ignore_index=True)
#
# cols = ("text", "aiGenerated")
# ai_part = pd.DataFrame(columns=cols)
# ai_part["text"] = ai_df["generated_text"]
# ai_part["aiGenerated"] = 1
# human_part = pd.DataFrame(columns=cols)
# human_part["text"] = human_df["text"]
# human_part["aiGenerated"] = 0
#
# df = pd.concat([ai_part, human_part]).sample(frac=1, random_state=RANDOM_SEED, ignore_index=True)
# df.to_csv("/content/drive/MyDrive/AITextDetection/processedDataset.csv", encoding="utf-8-sig", index=False)
# print(df.shape)


## 3. Preprocessing (lemmatize + tokenize, for the TF-IDF / BiLSTM models)

Same shekar pipeline the production API uses for the classic ML and BiLSTM
models. This is the slow, CPU-bound step — Colab's free tier usually only gives
you 2 vCPUs (versus 12 locally), so this can take a while. The result is cached
to Drive so you only pay this cost once.

(GPT-2 uses a *different*, lighter preprocessing step — see section 12.)


In [ ]:
from shekar import Lemmatizer, Normalizer, SentenceTokenizer, WordTokenizer
from shekar.preprocessing import EmojiRemover, NonPersianRemover, PunctuationRemover

_pipeline = None

def _get_pipeline():
    global _pipeline
    if _pipeline is None:
        _pipeline = (
            EmojiRemover()
            | PunctuationRemover()
            | NonPersianRemover()
            | Normalizer()
            | Lemmatizer()
            | WordTokenizer()
        )
    return _pipeline

def tokenize(text: str) -> list[str]:
    text = re.sub(r"\n", " ", str(text))
    return list(_get_pipeline()(text))


In [ ]:
TOKEN_CACHE_PATH = "/content/drive/MyDrive/AITextDetection/tokenized_texts_colab.joblib"

try:
    tokenized = joblib.load(TOKEN_CACHE_PATH)
    assert len(tokenized) == len(df)
    print(f"Loaded cached tokenization for {len(tokenized)} documents.")
except (FileNotFoundError, AssertionError):
    num_cores = multiprocessing.cpu_count()
    print(f"Tokenizing {len(df)} documents on {num_cores} cores (this is the slow part)...")
    tokenized = Parallel(n_jobs=num_cores)(
        delayed(tokenize)(text) for text in tqdm(df["text"])
    )
    joblib.dump(tokenized, TOKEN_CACHE_PATH)
    print("Cached to Drive for next time.")

cleaned_texts = [" ".join(tokens) for tokens in tokenized]


## 4. Train / validation / test split

Same seed and 80/20 test split the local `training/split.py` uses, so `test_idx`
lines up with the test set the currently-deployed models were scored on — the
metrics below are directly comparable to the ones already in `models.json`.

The 20% test set is touched **exactly once**, at the very end, for every model —
classic ML, BiLSTM, and GPT-2 alike. A separate validation slice (carved out of
the training portion) is what BiLSTM and GPT-2 use for early stopping — the local
versions of both accidentally used the test set for that, which is a big part of
why their reported accuracy didn't hold up on new text.


In [ ]:
TEST_SIZE = 0.2
VAL_SIZE = 0.1  # fraction of the train+val portion

idx = df.index.to_numpy()
y_all = df["aiGenerated"].to_numpy()

train_val_idx, test_idx = train_test_split(
    idx, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y_all
)
train_idx, val_idx = train_test_split(
    train_val_idx,
    test_size=VAL_SIZE,
    random_state=RANDOM_SEED,
    stratify=y_all[train_val_idx],
)

print(f"train={len(train_idx)}  val={len(val_idx)}  test={len(test_idx)}")

y_train, y_val, y_test = y_all[train_idx], y_all[val_idx], y_all[test_idx]


## 5. TF-IDF features

The local vectorizer used **trigrams only** with no vocabulary cap, producing
9.3M dimensions — mostly rare 3-grams that let the linear/tree models memorize
training text instead of generalizing. This one mixes unigrams through trigrams
and caps the vocabulary, which is both more robust and far smaller (so the
resulting model files are much smaller too).


In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 3),
    max_features=150_000,
    min_df=2,
    max_df=0.9,
    sublinear_tf=True,
    strip_accents="unicode",
)

train_texts = [cleaned_texts[i] for i in train_idx]
tfidf_vectorizer.fit(train_texts)

X_all = tfidf_vectorizer.transform(cleaned_texts)
X_train, X_val, X_test = X_all[train_idx], X_all[val_idx], X_all[test_idx]
print("Vocabulary size:", len(tfidf_vectorizer.vocabulary_))
print(X_train.shape, X_val.shape, X_test.shape)

# Also build a train+val combined matrix for the final classic-model refits below
# (once hyperparameters are chosen via CV on train only, refit on train+val
# before the single held-out test evaluation).
trainval_idx = np.concatenate([train_idx, val_idx])
y_trainval = y_all[trainval_idx]
X_trainval = X_all[trainval_idx]


## 5b. Stylometric text-style features

Four hand-crafted features -- character entropy, sentence-length variety, n-gram repetition, and inter-sentence coherence -- get hstacked onto the TF-IDF matrix for every classic ML model below. **These formulas must stay byte-for-byte identical to `utils.compute_style_features` on the server** (see `utils.py`), or served predictions will silently diverge from what these models were trained on.

In [ ]:
def shannon_entropy(text: str) -> float:
    char_counts = Counter(text)
    total_chars = len(text)
    entropy = 0.0
    for count in char_counts.values():
        probability = count / total_chars
        entropy -= probability * math.log2(probability)
    return entropy


def sentence_variety(text: str) -> float:
    sentences = list(SentenceTokenizer().tokenize(text))
    if len(sentences) < 2:
        return 0.0
    sentence_lengths = [len(sentence.split()) for sentence in sentences]
    mean_length = float(np.mean(sentence_lengths))
    if mean_length == 0:
        return 0.0
    return min(float(np.std(sentence_lengths)) / mean_length, 1.0)


def pattern_repetition(text: str, n: int = 3) -> float:
    tokens = list(WordTokenizer().tokenize(text))
    if len(tokens) < n:
        return 0.0
    ngrams = zip(*[tokens[i:] for i in range(n)])
    ngram_counts = Counter(" ".join(ngram) for ngram in ngrams)
    if not ngram_counts:
        return 0.0
    repeated = sum(1 for count in ngram_counts.values() if count > 1)
    return repeated / len(ngram_counts)


def text_coherence(text: str, vectorizer) -> float:
    sentences = [s for s in SentenceTokenizer().tokenize(text) if s.strip()]
    if len(sentences) < 2:
        return 1.0
    matrix = vectorizer.transform(sentences)
    similarities = [
        float(cosine_similarity(matrix[i], matrix[i + 1])[0][0])
        for i in range(matrix.shape[0] - 1)
    ]
    return min(max(float(np.mean(similarities)), 0.0), 1.0)


STYLE_FEATURE_NAMES = ["entropy", "sentence_variety", "repetition", "coherence"]
STYLE_FEATURE_COUNT = len(STYLE_FEATURE_NAMES)


def compute_style_features(text: str, cleaned_text: str, vectorizer) -> list[float]:
    entropy = shannon_entropy(text)
    max_entropy = math.log2(max(len(set(cleaned_text.replace(" ", ""))), 2))
    return [
        min(entropy / max_entropy, 1.0),
        sentence_variety(text),
        pattern_repetition(text, n=3),
        text_coherence(text, vectorizer),
    ]


STYLE_CACHE_PATH = "/content/drive/MyDrive/AITextDetection/style_features_colab.joblib"

try:
    style_features = joblib.load(STYLE_CACHE_PATH)
    assert len(style_features) == len(df)
    print(f"Loaded cached style features for {len(style_features)} documents.")
except (FileNotFoundError, AssertionError):
    print(f"Computing style features for {len(df)} documents (uses `tfidf_vectorizer`, single-process)...")
    style_features = np.array([
        compute_style_features(str(text), cleaned, tfidf_vectorizer)
        for text, cleaned in tqdm(zip(df["text"], cleaned_texts), total=len(df))
    ])
    joblib.dump(style_features, STYLE_CACHE_PATH)
    print("Cached to Drive for next time.")

# Style columns always come after the TF-IDF columns -- TFIDF_DIM marks
# where they start, used below to keep them out of the tree models' chi2
# selection (they should always be kept, not compete for a chi2 slot).
TFIDF_DIM = X_all.shape[1]
X_all = hstack([X_all, csr_matrix(style_features)]).tocsr()
X_train, X_val, X_test = X_all[train_idx], X_all[val_idx], X_all[test_idx]
X_trainval = X_all[trainval_idx]
print("TF-IDF dim:", TFIDF_DIM, "| + style features ->", X_all.shape[1])


In [ ]:
results = {}  # model_id -> {"model": ..., "metrics": ..., "confusion": ..., "extra": {...}}

def evaluate(model, X, y):
    y_pred = model.predict(X)
    cm = confusion_matrix(y, y_pred)
    tn, fp, fn, tp = cm.ravel()
    metrics = {
        "accuracy": round(float(accuracy_score(y, y_pred)), 4),
        "precision": round(float(precision_score(y, y_pred)), 4),
        "recall": round(float(recall_score(y, y_pred)), 4),
        "f1Score": round(float(f1_score(y, y_pred)), 4),
    }
    confusion = {
        "trueNegative": int(tn),
        "falsePositive": int(fp),
        "falseNegative": int(fn),
        "truePositive": int(tp),
    }
    return metrics, confusion


## 6. SVM (LinearSVC)

In [ ]:
svm_search = RandomizedSearchCV(
    LinearSVC(random_state=RANDOM_SEED, max_iter=10_000, dual=True),
    param_distributions={
        "C": loguniform(1e-3, 1e2),
        "loss": ["hinge", "squared_hinge"],
        "class_weight": [None, "balanced"],
    },
    n_iter=15,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbose=1,
)
t0 = time.time()
svm_search.fit(X_train, y_train)
print(f"Best params: {svm_search.best_params_}  (searched in {time.time()-t0:.0f}s)")

svm_model = LinearSVC(random_state=RANDOM_SEED, max_iter=10_000, **svm_search.best_params_)
svm_model.fit(X_trainval, y_trainval)
metrics, confusion = evaluate(svm_model, X_test, y_test)
print(metrics)
results["svm"] = {"model": svm_model, "metrics": metrics, "confusion": confusion,
                   "extra": {"algorithm": "LinearSVC", "best_params": svm_search.best_params_}}


## 7. Logistic Regression

In [ ]:
logreg_search = RandomizedSearchCV(
    LogisticRegression(random_state=RANDOM_SEED, max_iter=3000),
    param_distributions={
        "C": loguniform(1e-3, 1e2),
        "solver": ["liblinear", "saga"],
        "class_weight": [None, "balanced"],
    },
    n_iter=15,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbose=1,
)
t0 = time.time()
logreg_search.fit(X_train, y_train)
print(f"Best params: {logreg_search.best_params_}  (searched in {time.time()-t0:.0f}s)")

logreg_model = LogisticRegression(random_state=RANDOM_SEED, max_iter=3000, **logreg_search.best_params_)
logreg_model.fit(X_trainval, y_trainval)
metrics, confusion = evaluate(logreg_model, X_test, y_test)
print(metrics)
results["logreg"] = {"model": logreg_model, "metrics": metrics, "confusion": confusion,
                      "extra": {"algorithm": "LogisticRegression", "best_params": logreg_search.best_params_}}


## 8. Naive Bayes

In [ ]:
nb_search = GridSearchCV(
    MultinomialNB(),
    param_grid={"alpha": [0.01, 0.05, 0.1, 0.3, 0.5, 1.0, 2.0]},
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=1,
)
nb_search.fit(X_train, y_train)
print("Best params:", nb_search.best_params_)

nb_model = MultinomialNB(**nb_search.best_params_)
nb_model.fit(X_trainval, y_trainval)
metrics, confusion = evaluate(nb_model, X_test, y_test)
print(metrics)
results["naive_bayes"] = {"model": nb_model, "metrics": metrics, "confusion": confusion,
                           "extra": {"algorithm": "MultinomialNB", "best_params": nb_search.best_params_}}


## 9. Random Forest

Trees don't scale to 150k dimensions, so this keeps the same chi2 feature
selection idea as the local pipeline, just with a larger budget (30k instead of
20k) and, importantly, **tuned tree depth / leaf size** — the local version grew
unlimited-depth trees, which is a classic overfitting recipe on noisy TF-IDF
features.


In [ ]:
TREE_FEATURE_COUNT = 30_000

# chi2-select TREE_FEATURE_COUNT columns out of the TF-IDF block only,
# always passing the style-feature columns through untouched (shared by
# both Random Forest and Gradient Boosting below).
tree_selector = ColumnTransformer(
    [
        ("tfidf_select", SelectKBest(chi2, k=TREE_FEATURE_COUNT), slice(0, TFIDF_DIM)),
        ("style", "passthrough", slice(TFIDF_DIM, None)),
    ],
    sparse_threshold=1.0,
)
tree_selector.fit(X_train, y_train)

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=1),
    param_distributions={
        "n_estimators": randint(200, 500),
        "max_depth": [15, 25, 40, None],
        "min_samples_leaf": randint(1, 8),
        "max_features": ["sqrt", "log2"],
        "class_weight": [None, "balanced"],
    },
    n_iter=12,
    cv=3,
    scoring="f1",
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbose=2,
)
t0 = time.time()
rf_search.fit(tree_selector.transform(X_train), y_train)
print(f"Best params: {rf_search.best_params_}  (searched in {time.time()-t0:.0f}s)")

rf_pipeline = Pipeline([
    ("select", tree_selector),
    ("clf", RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=-1, **rf_search.best_params_)),
])
rf_pipeline.fit(X_trainval, y_trainval)
metrics, confusion = evaluate(rf_pipeline, X_test, y_test)
print(metrics)
results["random_forest"] = {"model": rf_pipeline, "metrics": metrics, "confusion": confusion,
                             "extra": {"algorithm": "RandomForestClassifier", "best_params": rf_search.best_params_,
                                       "featureCount": TREE_FEATURE_COUNT + STYLE_FEATURE_COUNT}}


## 10. Gradient Boosting

In [ ]:
gb_search = RandomizedSearchCV(
    GradientBoostingClassifier(random_state=RANDOM_SEED),
    param_distributions={
        "n_estimators": randint(100, 350),
        "max_depth": [2, 3, 4],
        "learning_rate": loguniform(0.02, 0.25),
        "subsample": [0.7, 0.85, 1.0],
    },
    n_iter=10,
    cv=3,
    scoring="f1",
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbose=2,
)
t0 = time.time()
gb_search.fit(tree_selector.transform(X_train), y_train)
print(f"Best params: {gb_search.best_params_}  (searched in {time.time()-t0:.0f}s)")

gb_pipeline = Pipeline([
    ("select", tree_selector),
    ("clf", GradientBoostingClassifier(random_state=RANDOM_SEED, **gb_search.best_params_)),
])
gb_pipeline.fit(X_trainval, y_trainval)
metrics, confusion = evaluate(gb_pipeline, X_test, y_test)
print(metrics)
results["gradient_boosting"] = {"model": gb_pipeline, "metrics": metrics, "confusion": confusion,
                                 "extra": {"algorithm": "GradientBoostingClassifier", "best_params": gb_search.best_params_,
                                           "featureCount": TREE_FEATURE_COUNT + STYLE_FEATURE_COUNT}}


## 11. BiLSTM (GPU, from scratch)

Same architecture as the deployed model (so it's a drop-in replacement for
`utils.py` — no backend code changes needed), just trained properly this time:
bigger capacity, weight decay, gradient clipping, and **early stopping against
the validation set instead of the test set**.


In [ ]:
MAX_VOCAB_SIZE = 40_000
MIN_FREQ = 2
MAX_LEN = 256
EMBED_DIM = 200
HIDDEN_DIM = 192
BATCH_SIZE = 64
MAX_EPOCHS = 25
PATIENCE = 4
LR = 1e-3
WEIGHT_DECAY = 1e-5
PAD_IDX = 0
UNK_IDX = 1

train_tokens = [tokenized[i] for i in train_idx]
val_tokens = [tokenized[i] for i in val_idx]
test_tokens = [tokenized[i] for i in test_idx]

def build_vocab(token_lists):
    counter = Counter()
    for tokens in token_lists:
        counter.update(tokens)
    vocab = {"<pad>": PAD_IDX, "<unk>": UNK_IDX}
    for word, freq in counter.most_common(MAX_VOCAB_SIZE):
        if freq < MIN_FREQ:
            break
        vocab.setdefault(word, len(vocab))
    return vocab

vocab = build_vocab(train_tokens)
print("Vocab size:", len(vocab))

def encode(tokens, max_len=MAX_LEN):
    ids = [vocab.get(t, UNK_IDX) for t in tokens[:max_len]]
    if len(ids) < max_len:
        ids += [PAD_IDX] * (max_len - len(ids))
    return ids

def to_tensor(token_lists, labels):
    X = torch.tensor([encode(t) for t in token_lists], dtype=torch.long)
    y = torch.tensor(labels, dtype=torch.float32)
    return X, y

X_train_lstm, y_train_lstm = to_tensor(train_tokens, y_train)
X_val_lstm, y_val_lstm = to_tensor(val_tokens, y_val)
X_test_lstm, y_test_lstm = to_tensor(test_tokens, y_test)


In [ ]:
class TextDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]


class BiLSTMClassifier(nn.Module):
    # Must stay architecturally identical to utils.py's BiLSTMClassifier.

    def __init__(self, vocab_size, embed_dim, hidden_dim, pad_idx=PAD_IDX):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        emb = self.embedding(x)
        _, (h_n, _) = self.lstm(emb)
        h = torch.cat([h_n[-2], h_n[-1]], dim=1)
        h = self.dropout(h)
        return self.fc(h).squeeze(-1)


train_loader = DataLoader(TextDataset(X_train_lstm, y_train_lstm), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TextDataset(X_val_lstm, y_val_lstm), batch_size=256, shuffle=False)
test_loader = DataLoader(TextDataset(X_test_lstm, y_test_lstm), batch_size=256, shuffle=False)

lstm_model = BiLSTMClassifier(len(vocab), EMBED_DIM, HIDDEN_DIM).to(DEVICE)
optimizer = torch.optim.AdamW(lstm_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = nn.BCEWithLogitsLoss()

def run_eval(loader):
    lstm_model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for xb, yb in loader:
            probs = torch.sigmoid(lstm_model(xb.to(DEVICE)))
            all_probs.append(probs.cpu().numpy())
            all_labels.append(yb.numpy())
    probs = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)
    preds = (probs >= 0.5).astype(int)
    return f1_score(labels, preds), accuracy_score(labels, preds)

best_val_f1 = -1.0
best_state = None
epochs_without_improvement = 0

for epoch in range(1, MAX_EPOCHS + 1):
    t0 = time.time()
    lstm_model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = lstm_model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(lstm_model.parameters(), max_norm=2.0)
        optimizer.step()
        total_loss += loss.item() * xb.size(0)

    val_f1, val_acc = run_eval(val_loader)
    print(f"epoch {epoch}/{MAX_EPOCHS}  loss={total_loss/len(X_train_lstm):.4f}  "
          f"val_acc={val_acc:.4f}  val_f1={val_f1:.4f}  ({time.time()-t0:.0f}s)")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {k: v.clone() for k, v in lstm_model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f"Early stopping at epoch {epoch} (no val improvement for {PATIENCE} epochs).")
            break

lstm_model.load_state_dict(best_state)


In [ ]:
lstm_model.eval()
all_probs, all_labels = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        probs = torch.sigmoid(lstm_model(xb.to(DEVICE)))
        all_probs.append(probs.cpu().numpy())
        all_labels.append(yb.numpy())
probs = np.concatenate(all_probs)
preds = (probs >= 0.5).astype(int)

cm = confusion_matrix(y_test, preds)
tn, fp, fn, tp = cm.ravel()
lstm_metrics = {
    "accuracy": round(float(accuracy_score(y_test, preds)), 4),
    "precision": round(float(precision_score(y_test, preds)), 4),
    "recall": round(float(recall_score(y_test, preds)), 4),
    "f1Score": round(float(f1_score(y_test, preds)), 4),
}
lstm_confusion = {
    "trueNegative": int(tn), "falsePositive": int(fp),
    "falseNegative": int(fn), "truePositive": int(tp),
}
print(lstm_metrics)
results["bilstm"] = {
    "metrics": lstm_metrics,
    "confusion": lstm_confusion,
    "extra": {"algorithm": "BiLSTM", "vocab_size": len(vocab)},
}


## 12. GPT-2 — full fine-tune, and a better tokenizing approach

The local GPT-2 fine-tune was already decent (it builds on real pretrained
language understanding instead of learning everything from 21k rows), but it was
constrained by the same CPU limits as everything else: a 6,000/1,500 subsample
instead of the full dataset, only 64 tokens of context, only the last 4 of 12
transformer blocks unfrozen, and its "best" checkpoint was effectively chosen
by the single epoch it happened to finish rather than genuine validation. On a
Colab GPU none of those constraints apply, so this fine-tunes on the **full**
dataset, **all 12 blocks** unfrozen, **128 tokens** of context, with mixed
precision and proper validation-based early stopping.

### The tokenizing improvement

Two changes, both real fixes rather than tuning tweaks:

1. **Preprocessing.** The classic ML / BiLSTM pipeline above lemmatizes and
   word-tokenizes text — good for bag-of-words style features, actively harmful
   for a pretrained language model: lemmatization strips inflection/morphology
   the model was pretrained on and turns natural sentences into a token stream
   unlike anything in its pretraining data. For GPT-2 this notebook instead
   applies only shekar's `Normalizer` (Arabic→Persian character unification,
   digit/punctuation normalization, diacritic removal, repeated-letter and
   ZWNJ/spacing fixes) — enough to clean up the inconsistent Persian text people
   actually type, without destroying word forms.
2. **Tokenizer implementation.** The locally pickled `GPT2Tokenizer` is the slow,
   pure-Python implementation — on this machine it turned out to BPE-encode an
   entire multi-paragraph article before truncating to 64 tokens, which is why
   the first local fine-tuning attempt stalled for 20+ minutes with zero
   progress. `AutoTokenizer.from_pretrained("HooshvareLab/gpt2-fa")` pulls the
   fast (Rust-backed) tokenizer when the model repo provides one, which is both
   faster and doesn't have that pathology on long documents. This notebook
   exports that tokenizer alongside the model so the server can load it fully
   offline afterward — `utils.py` already prefers it automatically when the
   `gpt2_tokenizer/` folder is present (falling back to the old pickle
   otherwise), so no backend code changes are needed.


In [ ]:
from transformers import AutoTokenizer, GPT2LMHeadModel

GPT2_MODEL_NAME = "HooshvareLab/gpt2-fa"

gpt2_tokenizer = AutoTokenizer.from_pretrained(GPT2_MODEL_NAME)
gpt2_tokenizer.pad_token = "<pad>"
print("Using fast tokenizer:", gpt2_tokenizer.is_fast)

gpt2_lm = GPT2LMHeadModel.from_pretrained(GPT2_MODEL_NAME)
gpt2_backbone = gpt2_lm.transformer
gpt2_hidden_size = gpt2_backbone.config.n_embd
gpt2_n_layer = gpt2_backbone.config.n_layer
print(f"n_layer={gpt2_n_layer}  n_embd={gpt2_hidden_size}  vocab_size={gpt2_backbone.config.vocab_size}")


In [ ]:
from shekar import Normalizer as ShekarNormalizer

_gpt2_normalizer = ShekarNormalizer()

def normalize_for_gpt2(text: str) -> str:
    text = re.sub(r"\n", " ", str(text))
    return _gpt2_normalizer(text)

GPT2_NORM_CACHE_PATH = "/content/drive/MyDrive/AITextDetection/normalized_texts_gpt2.joblib"

try:
    normalized_texts = joblib.load(GPT2_NORM_CACHE_PATH)
    assert len(normalized_texts) == len(df)
    print(f"Loaded cached normalization for {len(normalized_texts)} documents.")
except (FileNotFoundError, AssertionError):
    print("Normalizing text for GPT-2 (character-level only, no lemmatization)...")
    normalized_texts = Parallel(n_jobs=multiprocessing.cpu_count())(
        delayed(normalize_for_gpt2)(t) for t in tqdm(df["text"])
    )
    joblib.dump(normalized_texts, GPT2_NORM_CACHE_PATH)
    print("Cached to Drive for next time.")


In [ ]:
GPT2_SEQ_LEN = 128
GPT2_BATCH_SIZE = 32
GPT2_MAX_EPOCHS = 4
GPT2_PATIENCE = 2
GPT2_LR_BACKBONE = 2e-5
GPT2_LR_HEAD = 1e-3
GPT2_UNFROZEN_BLOCKS = gpt2_n_layer  # fine-tune every transformer block

# Word/position embeddings stay frozen: fine-tuning every block already adapts
# the model plenty for a binary classification task, and freezing the ~32M-
# parameter token embedding table meaningfully lowers overfitting risk on 21k
# rows without hurting quality in practice.
for p in gpt2_backbone.parameters():
    p.requires_grad = False
for block in gpt2_backbone.h[gpt2_n_layer - GPT2_UNFROZEN_BLOCKS:]:
    for p in block.parameters():
        p.requires_grad = True
for p in gpt2_backbone.ln_f.parameters():
    p.requires_grad = True


class GPT2Classifier(nn.Module):
    # Must stay architecturally identical to utils.py's GPT2Classifier.

    def __init__(self, backbone, hidden_size):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        hidden = self.backbone(
            input_ids=input_ids, attention_mask=attention_mask
        ).last_hidden_state
        last_idx = attention_mask.sum(dim=1) - 1
        pooled = hidden[torch.arange(hidden.size(0)), last_idx]
        return self.head(pooled).squeeze(-1)


gpt2_model = GPT2Classifier(gpt2_backbone, gpt2_hidden_size).to(DEVICE)
trainable_backbone_params = [p for p in gpt2_backbone.parameters() if p.requires_grad]
n_trainable = sum(p.numel() for p in trainable_backbone_params) + sum(p.numel() for p in gpt2_model.head.parameters())
print(f"Trainable parameters: {n_trainable:,}")


In [ ]:
def gpt2_encode_batch(text_batch):
    enc = gpt2_tokenizer(
        list(text_batch),
        return_tensors="pt",
        truncation=True,
        max_length=GPT2_SEQ_LEN,
        padding="max_length",
    )
    return enc["input_ids"], enc["attention_mask"]


class GPT2TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = list(texts)
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]


def gpt2_collate(batch):
    texts, labels = zip(*batch)
    input_ids, attention_mask = gpt2_encode_batch(texts)
    return input_ids, attention_mask, torch.tensor(labels, dtype=torch.float32)


gpt2_train_texts = [normalized_texts[i] for i in train_idx]
gpt2_val_texts = [normalized_texts[i] for i in val_idx]
gpt2_test_texts = [normalized_texts[i] for i in test_idx]

gpt2_train_loader = DataLoader(
    GPT2TextDataset(gpt2_train_texts, y_train), batch_size=GPT2_BATCH_SIZE,
    shuffle=True, collate_fn=gpt2_collate,
)
gpt2_val_loader = DataLoader(
    GPT2TextDataset(gpt2_val_texts, y_val), batch_size=64,
    shuffle=False, collate_fn=gpt2_collate,
)
gpt2_test_loader = DataLoader(
    GPT2TextDataset(gpt2_test_texts, y_test), batch_size=64,
    shuffle=False, collate_fn=gpt2_collate,
)


In [ ]:
optimizer = torch.optim.AdamW([
    {"params": trainable_backbone_params, "lr": GPT2_LR_BACKBONE},
    {"params": gpt2_model.head.parameters(), "lr": GPT2_LR_HEAD},
])
criterion = nn.BCEWithLogitsLoss()
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

def gpt2_run_eval(loader):
    gpt2_model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for input_ids, attention_mask, labels in loader:
            with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
                probs = torch.sigmoid(gpt2_model(input_ids.to(DEVICE), attention_mask.to(DEVICE)))
            all_probs.append(probs.float().cpu().numpy())
            all_labels.append(labels.numpy())
    probs = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)
    preds = (probs >= 0.5).astype(int)
    return f1_score(labels, preds), accuracy_score(labels, preds)

best_val_f1 = -1.0
best_state = None
epochs_without_improvement = 0

for epoch in range(1, GPT2_MAX_EPOCHS + 1):
    t0 = time.time()
    gpt2_model.train()
    total_loss = 0.0
    for i, (input_ids, attention_mask, labels) in enumerate(gpt2_train_loader, start=1):
        input_ids, attention_mask, labels = input_ids.to(DEVICE), attention_mask.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
            logits = gpt2_model(input_ids, attention_mask)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(gpt2_model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * len(labels)
        if i % 50 == 0:
            print(f"  epoch {epoch} batch {i}/{len(gpt2_train_loader)} loss={loss.item():.4f} "
                  f"elapsed={time.time()-t0:.0f}s", flush=True)

    val_f1, val_acc = gpt2_run_eval(gpt2_val_loader)
    print(f"Epoch {epoch}/{GPT2_MAX_EPOCHS}  loss={total_loss/len(gpt2_train_texts):.4f}  "
          f"val_acc={val_acc:.4f}  val_f1={val_f1:.4f}  ({time.time()-t0:.0f}s)")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {k: v.clone() for k, v in gpt2_model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= GPT2_PATIENCE:
            print(f"Early stopping at epoch {epoch} (no val improvement for {GPT2_PATIENCE} epochs).")
            break

gpt2_model.load_state_dict(best_state)


In [ ]:
gpt2_model.eval()
all_probs, all_labels = [], []
with torch.no_grad():
    for input_ids, attention_mask, labels in gpt2_test_loader:
        with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
            probs = torch.sigmoid(gpt2_model(input_ids.to(DEVICE), attention_mask.to(DEVICE)))
        all_probs.append(probs.float().cpu().numpy())
        all_labels.append(labels.numpy())
probs = np.concatenate(all_probs)
preds = (probs >= 0.5).astype(int)

cm = confusion_matrix(y_test, preds)
tn, fp, fn, tp = cm.ravel()
gpt2_metrics = {
    "accuracy": round(float(accuracy_score(y_test, preds)), 4),
    "precision": round(float(precision_score(y_test, preds)), 4),
    "recall": round(float(recall_score(y_test, preds)), 4),
    "f1Score": round(float(f1_score(y_test, preds)), 4),
}
gpt2_confusion = {
    "trueNegative": int(tn), "falsePositive": int(fp),
    "falseNegative": int(fn), "truePositive": int(tp),
}
print(gpt2_metrics)
results["gpt2_transformer"] = {
    "metrics": gpt2_metrics,
    "confusion": gpt2_confusion,
    "extra": {"algorithm": "GPT2ForSequenceClassification (full fine-tune)"},
}


## 13. ParsBERT — bidirectional encoder fine-tune

The error analysis in this project's report found that **BiLSTM** — which
learns its own word embeddings from scratch on only ~17k documents — is the
model most exposed to distribution shift: it confidently misclassified a
casual/conversational AI-written text that the training data under-represents
(see `report/report.tex`, section "تحلیل خطای هدفمند BiLSTM"). GPT-2 is far
more robust because it starts from real pretrained Persian language knowledge,
but it's a **causal** (unidirectional) decoder — not architecturally built for
classification, just repurposed for it via last-token pooling.

This section fine-tunes **ParsBERT**
(`HooshvareLab/bert-fa-base-uncased`)~\cite{farahani2021parsbert}, a
**bidirectional** BERT-style encoder pretrained on a large Persian corpus, as
a fourth deep-learning model: the same transfer-learning benefit as GPT-2,
but with an architecture purpose-built for sequence classification (every
token attends to the *whole* sequence, and a dedicated `[CLS]` token is
trained to summarize it) instead of a decoder repurposed for the job.

It reuses the same normalized (not lemmatized) texts and train/val/test split
built for GPT-2 above — see section~\ref{sec:gpt2-preprocess} of the report
for why lemmatizing would hurt a model pretrained on natural word forms; the
same reasoning applies to ParsBERT's own WordPiece tokenizer. Thanks to
`utils.py`'s runtime-dispatch design (a `runtime` field in `models.json`
picks the loader/predictor), adding this model needs no backend architecture
change — only a new `pytorch-parsbert` case alongside the existing
`sklearn-tfidf` / `pytorch-lstm` / `pytorch-gpt2` ones.

In [ ]:
from transformers import AutoTokenizer as _AutoTokenizer, AutoModel

PARSBERT_MODEL_NAME = "HooshvareLab/bert-fa-base-uncased"

parsbert_tokenizer = _AutoTokenizer.from_pretrained(PARSBERT_MODEL_NAME)
parsbert_backbone = AutoModel.from_pretrained(PARSBERT_MODEL_NAME)
parsbert_hidden_size = parsbert_backbone.config.hidden_size
print(f"hidden_size={parsbert_hidden_size}  vocab_size={parsbert_backbone.config.vocab_size}")

In [ ]:
PARSBERT_SEQ_LEN = 256
PARSBERT_BATCH_SIZE = 32
PARSBERT_MAX_EPOCHS = 4
PARSBERT_PATIENCE = 2
PARSBERT_LR_BACKBONE = 2e-5
PARSBERT_LR_HEAD = 1e-3

# ParsBERT-base is 12 layers / ~110M params, comparable in size to GPT-2-fa,
# so the same "unfreeze every encoder layer, freeze only the (larger)
# embedding table" strategy from the GPT-2 section above applies here too.
for p in parsbert_backbone.parameters():
    p.requires_grad = False
for layer in parsbert_backbone.encoder.layer:
    for p in layer.parameters():
        p.requires_grad = True
for p in parsbert_backbone.pooler.parameters():
    p.requires_grad = True


class ParsBERTClassifier(nn.Module):
    # Must stay architecturally identical to utils.py's ParsBERTClassifier.

    def __init__(self, backbone, hidden_size):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        # [CLS] is always token position 0 for BERT-style encoders -- unlike
        # GPT-2's causal attention, every position here already attends to
        # the whole sequence, so no attention-mask-based index lookup is
        # needed to find "the token that saw everything".
        hidden = self.backbone(
            input_ids=input_ids, attention_mask=attention_mask
        ).last_hidden_state
        cls_repr = hidden[:, 0]
        return self.head(cls_repr).squeeze(-1)


parsbert_model = ParsBERTClassifier(parsbert_backbone, parsbert_hidden_size).to(DEVICE)
parsbert_trainable_backbone_params = [p for p in parsbert_backbone.parameters() if p.requires_grad]
n_trainable = (
    sum(p.numel() for p in parsbert_trainable_backbone_params)
    + sum(p.numel() for p in parsbert_model.head.parameters())
)
print(f"Trainable parameters: {n_trainable:,}")

In [ ]:
def parsbert_encode_batch(text_batch):
    enc = parsbert_tokenizer(
        list(text_batch),
        return_tensors="pt",
        truncation=True,
        max_length=PARSBERT_SEQ_LEN,
        padding="max_length",
    )
    return enc["input_ids"], enc["attention_mask"]


class ParsBERTTextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = list(texts)
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]


def parsbert_collate(batch):
    texts, labels = zip(*batch)
    input_ids, attention_mask = parsbert_encode_batch(texts)
    return input_ids, attention_mask, torch.tensor(labels, dtype=torch.float32)


# Reuses the same character-normalized (not lemmatized) texts built for GPT-2
# above -- ParsBERT's own WordPiece tokenizer expects natural Persian text
# too, for the same reason GPT-2's BPE tokenizer does.
parsbert_train_texts = gpt2_train_texts
parsbert_val_texts = gpt2_val_texts
parsbert_test_texts = gpt2_test_texts

parsbert_train_loader = DataLoader(
    ParsBERTTextDataset(parsbert_train_texts, y_train), batch_size=PARSBERT_BATCH_SIZE,
    shuffle=True, collate_fn=parsbert_collate,
)
parsbert_val_loader = DataLoader(
    ParsBERTTextDataset(parsbert_val_texts, y_val), batch_size=64,
    shuffle=False, collate_fn=parsbert_collate,
)
parsbert_test_loader = DataLoader(
    ParsBERTTextDataset(parsbert_test_texts, y_test), batch_size=64,
    shuffle=False, collate_fn=parsbert_collate,
)

In [ ]:
optimizer = torch.optim.AdamW([
    {"params": parsbert_trainable_backbone_params, "lr": PARSBERT_LR_BACKBONE},
    {"params": parsbert_model.head.parameters(), "lr": PARSBERT_LR_HEAD},
])
criterion = nn.BCEWithLogitsLoss()
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

def parsbert_run_eval(loader):
    parsbert_model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for input_ids, attention_mask, labels in loader:
            with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
                probs = torch.sigmoid(parsbert_model(input_ids.to(DEVICE), attention_mask.to(DEVICE)))
            all_probs.append(probs.float().cpu().numpy())
            all_labels.append(labels.numpy())
    probs = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)
    preds = (probs >= 0.5).astype(int)
    return f1_score(labels, preds), accuracy_score(labels, preds)

best_val_f1 = -1.0
best_state = None
epochs_without_improvement = 0

for epoch in range(1, PARSBERT_MAX_EPOCHS + 1):
    t0 = time.time()
    parsbert_model.train()
    total_loss = 0.0
    for i, (input_ids, attention_mask, labels) in enumerate(parsbert_train_loader, start=1):
        input_ids, attention_mask, labels = input_ids.to(DEVICE), attention_mask.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
            logits = parsbert_model(input_ids, attention_mask)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(parsbert_model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * len(labels)
        if i % 50 == 0:
            print(f"  epoch {epoch} batch {i}/{len(parsbert_train_loader)} loss={loss.item():.4f} "
                  f"elapsed={time.time()-t0:.0f}s", flush=True)

    val_f1, val_acc = parsbert_run_eval(parsbert_val_loader)
    print(f"Epoch {epoch}/{PARSBERT_MAX_EPOCHS}  loss={total_loss/len(parsbert_train_texts):.4f}  "
          f"val_acc={val_acc:.4f}  val_f1={val_f1:.4f}  ({time.time()-t0:.0f}s)")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {k: v.clone() for k, v in parsbert_model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PARSBERT_PATIENCE:
            print(f"Early stopping at epoch {epoch} (no val improvement for {PARSBERT_PATIENCE} epochs).")
            break

parsbert_model.load_state_dict(best_state)

In [ ]:
parsbert_model.eval()
all_probs, all_labels = [], []
with torch.no_grad():
    for input_ids, attention_mask, labels in parsbert_test_loader:
        with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
            probs = torch.sigmoid(parsbert_model(input_ids.to(DEVICE), attention_mask.to(DEVICE)))
        all_probs.append(probs.float().cpu().numpy())
        all_labels.append(labels.numpy())
probs = np.concatenate(all_probs)
preds = (probs >= 0.5).astype(int)

cm = confusion_matrix(y_test, preds)
tn, fp, fn, tp = cm.ravel()
parsbert_metrics = {
    "accuracy": round(float(accuracy_score(y_test, preds)), 4),
    "precision": round(float(precision_score(y_test, preds)), 4),
    "recall": round(float(recall_score(y_test, preds)), 4),
    "f1Score": round(float(f1_score(y_test, preds)), 4),
}
parsbert_confusion = {
    "trueNegative": int(tn), "falsePositive": int(fp),
    "falseNegative": int(fn), "truePositive": int(tp),
}
print(parsbert_metrics)
results["parsbert"] = {
    "metrics": parsbert_metrics,
    "confusion": parsbert_confusion,
    "extra": {"algorithm": "BertForSequenceClassification (full fine-tune)"},
}

## 14. Summary

In [ ]:
for model_id, r in results.items():
    print(model_id, r["metrics"])


## 15. Save everything and build the `models.json` fragment

This writes the retrained artifacts plus a JSON fragment using the exact schema
and ids `models.json` already uses (`svm`, `logreg`, `naive_bayes`,
`random_forest`, `gradient_boosting`, `bilstm`, `gpt2_transformer`, `parsbert`),
and the exact `runtime` values `utils.py` dispatches on (`sklearn-tfidf` /
`pytorch-lstm` / `pytorch-gpt2` / `pytorch-parsbert`) — so no backend
architecture changes are needed, only the new `pytorch-parsbert` case added
to `utils.py`'s `_LOADERS`/`_PREDICTORS` dispatch tables alongside this
notebook, plus replacing files and merging the JSON.


In [ ]:
OUT_DIR = "/content/retrained"
import os
os.makedirs(OUT_DIR, exist_ok=True)

joblib.dump(tfidf_vectorizer, f"{OUT_DIR}/tfidf_vectorizer.joblib")
joblib.dump(results["svm"]["model"], f"{OUT_DIR}/svModel.joblib")
joblib.dump(results["logreg"]["model"], f"{OUT_DIR}/logregModel.joblib")
joblib.dump(results["naive_bayes"]["model"], f"{OUT_DIR}/nbModel.joblib")
joblib.dump(results["random_forest"]["model"], f"{OUT_DIR}/rfModel.joblib")
joblib.dump(results["gradient_boosting"]["model"], f"{OUT_DIR}/gbModel.joblib")

torch.save(
    {
        "state_dict": lstm_model.state_dict(),
        "vocab": vocab,
        "config": {
            "embed_dim": EMBED_DIM,
            "hidden_dim": HIDDEN_DIM,
            "max_len": MAX_LEN,
            "pad_idx": PAD_IDX,
            "unk_idx": UNK_IDX,
        },
    },
    f"{OUT_DIR}/lstmModel.pt",
)

torch.save(
    {
        "state_dict": gpt2_model.state_dict(),
        "config": {
            "seq_len": GPT2_SEQ_LEN,
            "hidden_size": gpt2_hidden_size,
            "n_layer": gpt2_n_layer,
            "vocab_size": gpt2_backbone.config.vocab_size,
            "n_head": gpt2_backbone.config.n_head,
        },
    },
    f"{OUT_DIR}/gpt2ClassifierModel.pt",
)

# Export the fast tokenizer so the server can load it fully offline —
# utils.py automatically prefers this folder over the old slow-tokenizer pickle
# when it's present in the project root.
gpt2_tokenizer.save_pretrained(f"{OUT_DIR}/gpt2_tokenizer")

torch.save(
    {
        "state_dict": parsbert_model.state_dict(),
        "config": {
            "seq_len": PARSBERT_SEQ_LEN,
            "hidden_size": parsbert_hidden_size,
            # Full backbone config (num layers/heads/vocab size/etc.), so the
            # server can rebuild an identically-shaped BertModel purely from
            # this checkpoint -- no network access to the HF Hub needed at
            # inference time, matching how gpt2-fa.mdl works locally already.
            "backbone_config": parsbert_backbone.config.to_dict(),
        },
    },
    f"{OUT_DIR}/parsbertClassifierModel.pt",
)

# Export ParsBERT's own WordPiece tokenizer, mirroring the GPT-2 tokenizer
# export above -- utils.py loads it fully offline from this folder.
parsbert_tokenizer.save_pretrained(f"{OUT_DIR}/parsbert_tokenizer")

print("Saved model artifacts to", OUT_DIR)


In [ ]:
def training_info(feature_count):
    return {
        "totalSamples": len(train_idx) + len(val_idx) + len(test_idx),
        "trainSamples": len(train_idx) + len(val_idx),
        "testSamples": len(test_idx),
        "featureCount": feature_count,
        "testSplitRatio": TEST_SIZE,
        "randomSeed": RANDOM_SEED,
        "dataset": "persian_ai_generated_final.csv + blogs.csv",
    }

vocab_size = len(tfidf_vectorizer.vocabulary_)

fragment = [
    {
        "id": "svm",
        "name": "SVM",
        "nameFa": "ماشین بردار پشتیبان",
        "description": "Linear Support Vector Machine (LinearSVC), hyperparameter-tuned via cross-validation on a unigram-to-trigram TF-IDF space, trained on Persian text to distinguish AI-generated content from human writing. Augmented with 4 stylometric text-style features (entropy, sentence variety, repetition, coherence).",
        "descriptionFa": "ماشین بردار پشتیبان خطی با تنظیم هایپرپارامتر از طریق اعتبارسنجی متقابل روی فضای TF-IDF یک‌گرم تا سه‌گرم، آموزش‌دیده برای تشخیص متن تولیدشده توسط هوش مصنوعی از نوشته انسان. همراه با ۴ ویژگی سبک‌شناسی متن (آنتروپی، تنوع جملات، تکرار، انسجام).",
        "algorithm": "LinearSVC",
        "type": "ml",
        "version": "3.0",
        "filename": "svModel.joblib",
        "runtime": "sklearn-tfidf",
        "ensembleWeight": 1.0,
        "vectorizer": "tfidf_vectorizer.joblib",
        "metrics": results["svm"]["metrics"],
        "confusionMatrix": results["svm"]["confusion"],
        "training": training_info(vocab_size + STYLE_FEATURE_COUNT),
    },
    {
        "id": "logreg",
        "name": "Logistic Regression",
        "nameFa": "رگرسیون لجستیک",
        "description": "Logistic Regression, hyperparameter-tuned via cross-validation on a unigram-to-trigram TF-IDF space, trained on Persian text to distinguish AI-generated content from human writing. Augmented with 4 stylometric text-style features (entropy, sentence variety, repetition, coherence).",
        "descriptionFa": "رگرسیون لجستیک با تنظیم هایپرپارامتر از طریق اعتبارسنجی متقابل روی فضای TF-IDF یک‌گرم تا سه‌گرم، آموزش‌دیده برای تشخیص متن تولیدشده توسط هوش مصنوعی از نوشته انسان. همراه با ۴ ویژگی سبک‌شناسی متن (آنتروپی، تنوع جملات، تکرار، انسجام).",
        "algorithm": "LogisticRegression",
        "type": "ml",
        "version": "3.0",
        "filename": "logregModel.joblib",
        "runtime": "sklearn-tfidf",
        "ensembleWeight": 1.0,
        "vectorizer": "tfidf_vectorizer.joblib",
        "metrics": results["logreg"]["metrics"],
        "confusionMatrix": results["logreg"]["confusion"],
        "training": training_info(vocab_size + STYLE_FEATURE_COUNT),
    },
    {
        "id": "naive_bayes",
        "name": "Naive Bayes",
        "nameFa": "بیز ساده",
        "description": "Multinomial Naive Bayes, alpha-tuned via cross-validation on a unigram-to-trigram TF-IDF space, trained on Persian text to distinguish AI-generated content from human writing. Augmented with 4 stylometric text-style features (entropy, sentence variety, repetition, coherence).",
        "descriptionFa": "بیز ساده چندجمله‌ای با تنظیم آلفا از طریق اعتبارسنجی متقابل روی فضای TF-IDF یک‌گرم تا سه‌گرم، آموزش‌دیده برای تشخیص متن تولیدشده توسط هوش مصنوعی از نوشته انسان. همراه با ۴ ویژگی سبک‌شناسی متن (آنتروپی، تنوع جملات، تکرار، انسجام).",
        "algorithm": "MultinomialNB",
        "type": "ml",
        "version": "3.0",
        "filename": "nbModel.joblib",
        "runtime": "sklearn-tfidf",
        "ensembleWeight": 1.0,
        "vectorizer": "tfidf_vectorizer.joblib",
        "metrics": results["naive_bayes"]["metrics"],
        "confusionMatrix": results["naive_bayes"]["confusion"],
        "training": training_info(vocab_size + STYLE_FEATURE_COUNT),
    },
    {
        "id": "random_forest",
        "name": "Random Forest",
        "nameFa": "جنگل تصادفی",
        "description": "Random Forest, hyperparameter-tuned (including bounded tree depth to fight overfitting) via cross-validation on the top 30,000 chi2-selected TF-IDF features, trained on Persian text to distinguish AI-generated content from human writing. Augmented with 4 stylometric text-style features (entropy, sentence variety, repetition, coherence).",
        "descriptionFa": "جنگل تصادفی با تنظیم هایپرپارامتر (از جمله محدودسازی عمق درخت برای کاهش بیش‌برازش) روی ۳۰٬۰۰۰ ویژگی برتر TF-IDF (انتخاب‌شده با chi2)، آموزش‌دیده برای تشخیص متن تولیدشده توسط هوش مصنوعی از نوشته انسان. همراه با ۴ ویژگی سبک‌شناسی متن (آنتروپی، تنوع جملات، تکرار، انسجام).",
        "algorithm": "RandomForestClassifier",
        "type": "ml",
        "version": "3.0",
        "filename": "rfModel.joblib",
        "runtime": "sklearn-tfidf",
        "ensembleWeight": 1.0,
        "vectorizer": "tfidf_vectorizer.joblib",
        "metrics": results["random_forest"]["metrics"],
        "confusionMatrix": results["random_forest"]["confusion"],
        "training": training_info(results["random_forest"]["extra"]["featureCount"]),
    },
    {
        "id": "gradient_boosting",
        "name": "Gradient Boosting",
        "nameFa": "گرادیان بوستینگ",
        "description": "Gradient Boosting, hyperparameter-tuned via cross-validation on the top 30,000 chi2-selected TF-IDF features, trained on Persian text to distinguish AI-generated content from human writing. Augmented with 4 stylometric text-style features (entropy, sentence variety, repetition, coherence).",
        "descriptionFa": "گرادیان بوستینگ با تنظیم هایپرپارامتر از طریق اعتبارسنجی متقابل روی ۳۰٬۰۰۰ ویژگی برتر TF-IDF (انتخاب‌شده با chi2)، آموزش‌دیده برای تشخیص متن تولیدشده توسط هوش مصنوعی از نوشته انسان. همراه با ۴ ویژگی سبک‌شناسی متن (آنتروپی، تنوع جملات، تکرار، انسجام).",
        "algorithm": "GradientBoostingClassifier",
        "type": "ml",
        "version": "3.0",
        "filename": "gbModel.joblib",
        "runtime": "sklearn-tfidf",
        "ensembleWeight": 1.0,
        "vectorizer": "tfidf_vectorizer.joblib",
        "metrics": results["gradient_boosting"]["metrics"],
        "confusionMatrix": results["gradient_boosting"]["confusion"],
        "training": training_info(results["gradient_boosting"]["extra"]["featureCount"]),
    },
    {
        "id": "bilstm",
        "name": "BiLSTM",
        "nameFa": "شبکه دوسویه LSTM",
        "description": "A bidirectional LSTM with its own learned word embeddings, trained from scratch on GPU with early stopping against a held-out validation set (not the test set), on lemmatized Persian text, to distinguish AI-generated content from human writing.",
        "descriptionFa": "شبکه عصبی LSTM دوسویه با embedding آموخته‌شده اختصاصی، آموزش‌دیده از ابتدا روی GPU با early stopping بر پایه مجموعه اعتبارسنجی جداگانه (نه مجموعه آزمون)، روی متن فارسی نرمال‌شده و ریشه‌یابی‌شده، برای تشخیص متن تولیدشده توسط هوش مصنوعی از نوشته انسان.",
        "algorithm": "BiLSTM",
        "type": "dl",
        "version": "2.0",
        "filename": "lstmModel.pt",
        "runtime": "pytorch-lstm",
        "ensembleWeight": 1.5,
        "metrics": results["bilstm"]["metrics"],
        "confusionMatrix": results["bilstm"]["confusion"],
        "training": training_info(results["bilstm"]["extra"]["vocab_size"]),
    },
    {
        "id": "gpt2_transformer",
        "name": "GPT-2 Transformer (fine-tuned)",
        "nameFa": "ترنسفورمر GPT-2 (فاین‌تیون‌شده)",
        "description": "HooshvareLab/gpt2-fa (a 12-layer, 124M-parameter Persian GPT-2) fully fine-tuned (all 12 transformer blocks) on the full training set with 128 tokens of context, GPU + mixed precision, and validation-based early stopping. Text is normalized (not lemmatized) before tokenizing with GPT-2's own fast tokenizer, to match its pretraining distribution.",
        "descriptionFa": "مدل GPT-2 فارسی (HooshvareLab/gpt2-fa) با ۱۲ لایه و ۱۲۴ میلیون پارامتر که تمام ۱۲ بلوک ترنسفورمر آن روی کل مجموعه آموزشی با ۱۲۸ توکن زمینه، GPU و دقت مختلط، و early stopping بر پایه مجموعه اعتبارسنجی فاین‌تیون شده است. متن پیش از توکن‌سازی با توکنایزر سریع خود GPT-2 تنها نرمال‌سازی می‌شود (نه ریشه‌یابی) تا با توزیع پیش‌آموزش مدل سازگار بماند.",
        "algorithm": "GPT2ForSequenceClassification (full fine-tune)",
        "type": "dl",
        "version": "2.0",
        "filename": "gpt2ClassifierModel.pt",
        "runtime": "pytorch-gpt2",
        "ensembleWeight": 3.0,
        "metrics": results["gpt2_transformer"]["metrics"],
        "confusionMatrix": results["gpt2_transformer"]["confusion"],
        "training": {
            "totalSamples": len(train_idx) + len(val_idx) + len(test_idx),
            "trainSamples": len(train_idx) + len(val_idx),
            "testSamples": len(test_idx),
            "featureCount": gpt2_hidden_size,
            "testSplitRatio": TEST_SIZE,
            "randomSeed": RANDOM_SEED,
            "dataset": "persian_ai_generated_final.csv + blogs.csv",
        },
    },
    {
        "id": "parsbert",
        "name": "ParsBERT Transformer (fine-tuned)",
        "nameFa": "ترنسفورمر ParsBERT (فاین‌تیون‌شده)",
        "description": "HooshvareLab/bert-fa-base-uncased (a 12-layer, ~110M-parameter Persian BERT) fully fine-tuned (all 12 encoder layers) on the full training set with 256 tokens of context, GPU + mixed precision, and validation-based early stopping. Uses the bidirectional [CLS] token representation instead of GPT-2's last-token pooling, and its own WordPiece tokenizer on lightly normalized (not lemmatized) text.",
        "descriptionFa": "مدل ParsBERT فارسی (HooshvareLab/bert-fa-base-uncased) با ۱۲ لایه و حدود ۱۱۰ میلیون پارامتر که تمام ۱۲ لایهٔ رمزگذار آن روی کل مجموعه آموزشی با ۲۵۶ توکن زمینه، GPU و دقت مختلط، و early stopping بر پایه مجموعه اعتبارسنجی فاین‌تیون شده است. برخلاف GPT-2، از بازنمایی دوسویهٔ توکن [CLS] و توکن‌ساز WordPiece اختصاصی خود روی متن نرمال‌شده (نه ریشه‌یابی‌شده) استفاده می‌کند.",
        "algorithm": "BertForSequenceClassification (full fine-tune)",
        "type": "dl",
        "version": "1.0",
        "filename": "parsbertClassifierModel.pt",
        "runtime": "pytorch-parsbert",
        # Provisional -- adjust once real metrics are in; GPT-2 (3.0) is the
        # current best model and BiLSTM (1.5) the most distribution-shift-
        # sensitive one, so 2.0 is a reasonable prior until this is measured.
        "ensembleWeight": 2.0,
        "metrics": results["parsbert"]["metrics"],
        "confusionMatrix": results["parsbert"]["confusion"],
        "training": {
            "totalSamples": len(train_idx) + len(val_idx) + len(test_idx),
            "trainSamples": len(train_idx) + len(val_idx),
            "testSamples": len(test_idx),
            "featureCount": parsbert_hidden_size,
            "testSplitRatio": TEST_SIZE,
            "randomSeed": RANDOM_SEED,
            "dataset": "persian_ai_generated_final.csv + blogs.csv",
        },
    },
]

with open(f"{OUT_DIR}/models_colab_update.json", "w", encoding="utf-8") as f:
    json.dump(fragment, f, ensure_ascii=False, indent=2)

print(json.dumps(fragment, ensure_ascii=False, indent=2))


## 16. Download

Zips everything in `/content/retrained` (models + `tfidf_vectorizer.joblib` +
the exported `gpt2_tokenizer/` and `parsbert_tokenizer/` folders +
`models_colab_update.json`) and downloads it.

**To apply it locally:**
1. Unzip into the project root, overwriting `tfidf_vectorizer.joblib`,
   `svModel.joblib`, `logregModel.joblib`, `nbModel.joblib`, `rfModel.joblib`,
   `gbModel.joblib`, `lstmModel.pt`, `gpt2ClassifierModel.pt`,
   `parsbertClassifierModel.pt`, and adding the `gpt2_tokenizer/` and
   `parsbert_tokenizer/` folders alongside them.
2. Open `models_colab_update.json` and replace the matching entries (`svm`,
   `logreg`, `naive_bayes`, `random_forest`, `gradient_boosting`, `bilstm`,
   `gpt2_transformer`, `parsbert`) in `models.json` with these new ones
   (`parsbert` is a brand-new entry, not a replacement).
3. Pull the `pytorch-parsbert` loader/predictor added to `utils.py` (see the
   accompanying code change) — unlike the other rows in this list, this one
   genuinely needs a backend code change, since `parsbert` is a new runtime,
   not a retrain of an existing one.
4. Restart `server.py`. `utils.py` already prefers the exported
   `gpt2_tokenizer/`/`parsbert_tokenizer/` folders automatically when
   present, so no further code changes are needed.

Note: the TF-IDF vectorizer's vocabulary changed, so **every** `sklearn-tfidf`
model must come from this same run — don't mix an old model file with the new
`tfidf_vectorizer.joblib` or vice versa.


In [ ]:
import shutil
from google.colab import files

zip_path = shutil.make_archive("/content/retrained_models", "zip", OUT_DIR)
files.download(zip_path)
